# Explainable ML-Assisted Design-Space Exploration of Steel SMRFs

Publication-ready analysis notebook for the EESD resubmission. The notebook is organized to support reviewer-ready reproducibility: data loading, fixed train/test split, model training, model evaluation, centralized geometry-conditioned sampling, explainability, Pareto exploration, and interpretable rule extraction.

**Key cleanup relative to the working notebook:** all synthetic design-space sampling is now handled in one centralized function, `make_geometry_conditioned_samples()`. SHAP/ALE/Sobol/tradeoff/Pareto analyses use the same geometry-conditioned sampling logic so that the workflow is transparent and reproducible.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
import os
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

try:
    import umap
except ImportError:
    umap = None
    print("Optional package 'umap-learn' is not installed; UMAP-dependent cells will be skipped.")
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility and output folders
# ------------------------------------------------------------
RANDOM_STATE = 42
DATA_DIR = Path(".")
FIGURE_DIR = Path("figures")
RESULT_DIR = Path("results")
MODEL_DIR = Path("models")

for folder in [FIGURE_DIR, RESULT_DIR, MODEL_DIR]:
    folder.mkdir(exist_ok=True)

# Main objectives minimized in the design-space and Pareto analyses
TARGET_COLS = ["Functional_DBE", "Functional_MCE", "EAL"]


## 1. Load and merge structural-design and performance data

Expected input files in the repository root:
- `BuildingData.csv`
- `PerformanceData.csv`
- `fixed_train_test_split.csv` (optional; created automatically if missing)

In [ ]:
# ============================================================
# 2. Load data
# ============================================================
building_path = DATA_DIR / "BuildingData.csv"
recovery_path = DATA_DIR / "PerformanceData.csv"

building_df = pd.read_csv(building_path, encoding="latin1")
recovery_df = pd.read_csv(recovery_path)

building_df.columns = building_df.columns.str.replace("\xa0", " ", regex=False).str.strip()
recovery_df.columns = recovery_df.columns.str.replace("\xa0", " ", regex=False).str.strip()

building_df = building_df.dropna(axis=1, how="all")
recovery_df = recovery_df.dropna(axis=1, how="all")

id_col = "Building ID"

target_cols = TARGET_COLS

data = building_df.merge(
    recovery_df[[id_col] + target_cols],
    on=id_col,
    how="inner",
    validate="one_to_one"
)

X = data.drop(columns=[id_col] + target_cols)
X = X.select_dtypes(include=[np.number])

building_ids = data[id_col]

print("Merged data shape:", data.shape)
print("Feature matrix shape:", X.shape)
print("Targets:", target_cols)

## 2. Reproducible train/test split

The split is saved by Building ID so reviewers can reproduce the same train/test partition.

In [ ]:
# ============================================================
# 3. Fixed train-test split based on Building ID
# ============================================================
split_file = DATA_DIR / "fixed_train_test_split.csv"

if os.path.exists(split_file):
    split_df = pd.read_csv(split_file)
    train_ids = split_df.loc[split_df["Set"] == "Train", id_col]
    test_ids = split_df.loc[split_df["Set"] == "Test", id_col]

else:
    train_ids, test_ids = train_test_split(
        building_ids,
        test_size=0.20,
        random_state=42,
        shuffle=True
    )

    split_df = pd.DataFrame({
        id_col: list(train_ids) + list(test_ids),
        "Set": ["Train"] * len(train_ids) + ["Test"] * len(test_ids)
    })

    split_df.to_csv(split_file, index=False)

train_mask = building_ids.isin(train_ids)
test_mask = building_ids.isin(test_ids)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 3. Plotting and model-training helper functions

These functions are kept together so the scientific workflow below is easier to inspect and rerun.

In [ ]:
# ============================================================
# 4. Publication-quality predicted-vs-observed plot
# ============================================================

from matplotlib.ticker import ScalarFormatter

def plot_predicted_vs_observed(
    y_train_true,
    y_train_pred,
    y_test_true,
    y_test_pred,

    xlabel=None,
    ylabel=None,
    target_name=None,
    title=None,

    xlim=None,
    ylim=None,

    xticks=None,
    yticks=None,

    figsize=(7.2, 7.2),

    train_color="#009E73",   # Okabe-Ito green
    test_color="#E69F00",    # Okabe-Ito orange

    train_alpha=0.65,
    test_alpha=0.80,

    train_marker_size=55,
    test_marker_size=65,

    edgecolor="black",
    linewidth=0.7,

    font_family="Times New Roman",
    font_size=12,
    tick_size=10,

    dpi=300,

    save_path=None
):

    plt.rcParams["font.family"] = font_family

    # --------------------------------------------------------
    # Automatic labels
    # --------------------------------------------------------
    if target_name is not None:

        if target_name in ["Functional_DBE", "Functional_MCE"]:
            xlabel = "Observed recovery time (days)"
            ylabel = "Predicted recovery time (days)"

        elif target_name == "EAL":
            xlabel = r"Observed EAL ($\times10^{5}$ )"
            ylabel = r"Predicted EAL ($\times10^{5}$ )"

    if xlabel is None:
        xlabel = "Observed"

    if ylabel is None:
        ylabel = "Predicted"

    # --------------------------------------------------------
    # Axis limits
    # --------------------------------------------------------
    all_true = np.concatenate([y_train_true, y_test_true])
    all_pred = np.concatenate([y_train_pred, y_test_pred])

    max_value = max(all_true.max(), all_pred.max())

    if max_value <= 10:
        rounded_max = np.ceil(max_value)
    elif max_value <= 100:
        rounded_max = np.ceil(max_value / 10) * 10
    elif max_value <= 1000:
        rounded_max = np.ceil(max_value / 100) * 100
    elif max_value <= 10000:
        rounded_max = np.ceil(max_value / 1000) * 1000
    elif max_value <= 100000:
        rounded_max = np.ceil(max_value / 10000) * 10000
    else:
        rounded_max = np.ceil(max_value / 100000) * 100000

    if xlim is None:
        xlim = [0, rounded_max]

    if ylim is None:
        ylim = [0, rounded_max]

    # --------------------------------------------------------
    # Ticks
    # --------------------------------------------------------
    if xticks is None:
        xticks = np.linspace(0, rounded_max, 4)

    if yticks is None:
        yticks = np.linspace(0, rounded_max, 4)

    if target_name == "EAL":
        xticks = np.round(xticks, -3)
        yticks = np.round(yticks, -3)
    else:
        if rounded_max <= 10:
            xticks = np.round(xticks, 1)
            yticks = np.round(yticks, 1)
        elif rounded_max <= 100:
            xticks = np.round(xticks, 0)
            yticks = np.round(yticks, 0)
        elif rounded_max <= 1000:
            xticks = np.round(xticks, -1)
            yticks = np.round(yticks, -1)
        else:
            xticks = np.round(xticks, -2)
            yticks = np.round(yticks, -2)

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    ax.scatter(
        y_train_true,
        y_train_pred,
        color=train_color,
        alpha=train_alpha,
        s=train_marker_size,
        edgecolors=edgecolor,
        linewidths=linewidth,
        marker="s",
        label="Training"
    )

    ax.scatter(
        y_test_true,
        y_test_pred,
        color=test_color,
        alpha=test_alpha,
        s=test_marker_size,
        edgecolors=edgecolor,
        linewidths=linewidth,
        marker="o",
        label="Testing"
    )

    ax.plot(
        xlim,
        xlim,
        linestyle="--",
        color="black",
        linewidth=1.5
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    # --------------------------------------------------------
    # Axis number formatting
    # --------------------------------------------------------
    if target_name == "EAL":

        formatter = ScalarFormatter(useMathText=True)
        formatter.set_scientific(True)
        formatter.set_powerlimits((5, 5))   # force ×10^5
        formatter.set_useOffset(False)

        ax.xaxis.set_major_formatter(formatter)
        ax.yaxis.set_major_formatter(formatter)

        # Hide the automatic offset text
        ax.xaxis.offsetText.set_visible(False)
        ax.yaxis.offsetText.set_visible(False)
     

    else:

        ax.ticklabel_format(
            axis="both",
            style="plain",
            useOffset=False
        )

    ax.set_xlabel(xlabel, fontsize=font_size)
    ax.set_ylabel(ylabel, fontsize=font_size)

    if title is not None:
        ax.set_title(title, fontsize=font_size + 1)

    ax.tick_params(
        axis="both",
        labelsize=tick_size,
        direction="in"
    )

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.02, 0.98),
        frameon=False,
        fontsize=tick_size - 2,
        handletextpad=0.3,
        borderpad=0.2,
        labelspacing=0.3
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.7,
        alpha=0.7
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.1)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")

    plt.show()

In [ ]:
# ============================================================
# Feature names for publication-quality SHAP figures
# ============================================================
feature_name_map = {
    # Replace these with your exact Excel column names
    "bay width (BW), ft": "BW",
    "TimePeriod": r"$T_1$",
    "FloorArea (sqft)": r"$A_F$",
    "total building height (TBH), ft": "H",
    "Building FRAME weight (lbf)": r"$W_L$",
    "Total building weight (lbf)": r"$W_T$",
    "number of stories (NS)": "NS",
    "Avg_ColumnBeamRatio (AVG_SCWB)": r"$SCWB_{avg}$",
    "Total building height/first story height": r"$h_1/H$",
    "Avg_DgnStoryDrift": r"$IDR_{avg}$",
    "Design SCWB ratio": r"$SCWB_{Dsgn}$",
    "Avg_ColumnAxialDCRatio": r"$D/C_{C,ax}$",
    "Avg_ColumnFlexuralDCRatio": r"$D/C_{C,flex}$",
    "Avg_BeamShearDCRatio": r"$D/C_{B,Sh}$",
    "minimum moment of inertia for all internal columns (IINTMIN), in^4": r"$I_{min,C,int}$",
    "minimum moment of inertia for all external columns (IEXTMIN),in^4": r"$I_{min,C,ext}$",
    "average moment of inertia for all external columns (IEXT), in^4": r"$I_{avg,C,ext}$",
    "maximum moment of inertia for all external columns (IEXTMAX), in^4": r"$I_{max,C,ext}$",
    "average moment of inertia for all beams (IB), in^4": r"$I_{avg,B}$",
    "maximum moment of inertia for all beams (IBMAX), in^4": r"$I_{max,B}$",
    "minimum moment of inertia for all beams (IBMIN), in^4": r"$I_{min,B}$",
    "average moment of inertia for internal columns of building (IINT), in^4": r"$I_{avg,C,int}$",
    "maximum moment of inertia for internal columns (IINTMAX), in^4": r"$I_{max,C,int}$",
    "Avg_DblrPlte_Thickness": r"$DP_{t,avg}$"
    # Add more as needed
}

In [ ]:
# 5. Random Forest training with feature selection and tuning
# ============================================================
def train_random_forest_for_target(
    X_train,
    X_test,
    y_train,
    y_test,
    target_name,
    random_state=42,
    n_iter=100
):
    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),

        ("feature_selector", SelectFromModel(
            estimator=RandomForestRegressor(
                n_estimators=500,
                random_state=random_state,
                n_jobs=-1
            ),
            threshold="median"
        )),

        ("model", RandomForestRegressor(
            random_state=random_state,
            n_jobs=-1,
            bootstrap=True
        ))
    ])

    param_grid = {
        "feature_selector__threshold": [
            "mean",
            "median",
            0.005,
            0.01,
            0.02
        ],
        "model__n_estimators": [
            300, 500, 800, 1000, 1500
        ],
        "model__max_depth": [
            None, 5, 10, 15, 20, 30, 40
        ],
        "model__min_samples_split": [
            2, 4, 6, 8, 10, 15
        ],
        "model__min_samples_leaf": [
            1, 2, 3, 4, 5, 8
        ],
        "model__max_features": [
            "sqrt", "log2", 0.3, 0.5, 0.7, 1.0
        ],
        "model__bootstrap": [
            True
        ]
    }

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring="r2",
        cv=cv,
        random_state=random_state,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    

    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    results = {
      "Target": target_name,
      "Model": "Random Forest",

      "Train R2": r2_score(y_train, y_pred_train),
      "Train MAE": mean_absolute_error(y_train, y_pred_train),
      "Train RMSE": np.sqrt(mean_squared_error(y_train, y_pred_train)),

      "Test R2": r2_score(y_test, y_pred_test),
      "Test MAE": mean_absolute_error(y_test, y_pred_test),
      "Test RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
  
      "Best CV R2": search.best_score_,
      "Best Parameters": search.best_params_
    }


    results_df = pd.DataFrame([results])

    print("\nTarget:", target_name)
    print("Best parameters:")
    print(search.best_params_)
    display(results_df)

    return best_model, search, results_df, y_pred_train, y_pred_test

In [ ]:
# ============================================================
# 5B. Baseline model: Multiple Linear Regression
# with the same embedded feature-selection strategy
# ============================================================
def train_baseline_linear_regression(
    X_train,
    X_test,
    y_train,
    y_test,
    target_name,
    random_state=42
):
    baseline_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),

        ("feature_selector", SelectFromModel(
            estimator=RandomForestRegressor(
                n_estimators=500,
                random_state=random_state,
                n_jobs=-1
            ),
            threshold="median"
        )),

        ("model", LinearRegression())
    ])

    baseline_pipeline.fit(X_train, y_train)

    y_pred_train = baseline_pipeline.predict(X_train)
    y_pred_test = baseline_pipeline.predict(X_test)

    results = {
     "Target": target_name,
     "Model": "Multiple Linear Regression",

     "Train R2": r2_score(y_train, y_pred_train),
     "Train MAE": mean_absolute_error(y_train, y_pred_train),
     "Train RMSE": np.sqrt(mean_squared_error(y_train, y_pred_train)),

     "Test R2": r2_score(y_test, y_pred_test),
     "Test MAE": mean_absolute_error(y_test, y_pred_test),
     "Test RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),

     "Best CV R2": np.nan,
     "Best Parameters": "Baseline model with RF-based feature selection; no hyperparameter tuning"
    }



    results_df = pd.DataFrame([results])

    return baseline_pipeline, results_df, y_pred_train, y_pred_test

In [ ]:
# ============================================================
# 10. Publication-quality SHAP analysis
# ============================================================

import matplotlib.pyplot as plt
import shap

def plot_publication_shap(
    shap_values,
    X_selected_df,
    target,
    max_display=15,
    font_family="Times New Roman",
    font_size=12,
    dpi=300,
    figsize=(8.2, 5.6)
):
    plt.rcParams["font.family"] = font_family

    clean_target = (
        target.replace(" ", "_")
              .replace("(", "")
              .replace(")", "")
              .replace("/", "_")
    )

    # --------------------------------------------------------
    # SHAP violin plot: original SHAP colors
    # --------------------------------------------------------
    plt.figure(figsize=figsize, dpi=dpi)

    shap.summary_plot(
        shap_values,
        X_selected_df,
        plot_type="violin",
        max_display=max_display,
        plot_size=figsize,
        show=False
    )

    fig = plt.gcf()
    ax = plt.gca()

    ax.set_xlabel(
        "SHAP value",
        fontsize=font_size + 2,
        fontfamily=font_family
    )

    ax.tick_params(
        axis="both",
        labelsize=font_size + 4,
        direction="in"
    )

    ax.set_title("")

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

    # Format SHAP colorbar but remove High/Low tick labels
    for axis in fig.axes:
        if axis is not ax:
            axis.set_yticks([])
            axis.set_yticklabels([])

            axis.tick_params(
                labelsize=font_size + 2,
                direction="in"
            )

            axis.yaxis.label.set_size(font_size + 3)
            axis.yaxis.label.set_family(font_family)

    plt.subplots_adjust(
        left=0.34,
        right=0.92,
        top=0.96,
        bottom=0.16
    )

    plt.savefig(
        f"SHAP_summary_{clean_target}.png",
        dpi=dpi
    )

    plt.savefig(
        f"SHAP_summary_{clean_target}.pdf"
    )

    plt.show()

    # --------------------------------------------------------
    # Mean absolute SHAP bar plot
    # --------------------------------------------------------
    plt.figure(figsize=figsize, dpi=dpi)

    shap.summary_plot(
        shap_values,
        X_selected_df,
        plot_type="bar",
        max_display=max_display,
        plot_size=figsize,
        show=False
    )

    fig = plt.gcf()
    ax = plt.gca()

    for patch in ax.patches:
        patch.set_facecolor("#D9D9D9")
        patch.set_edgecolor("black")
        patch.set_linewidth(0.9)
        patch.set_alpha(1.0)

    ax.set_xlabel(
        "Mean absolute SHAP value",
        fontsize=font_size + 2,
        fontfamily=font_family
    )

    ax.tick_params(
        axis="both",
        labelsize=font_size + 4,
        direction="in"
    )

    ax.set_title("")

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

    plt.subplots_adjust(
        left=0.34,
        right=0.92,
        top=0.96,
        bottom=0.16
    )

    plt.savefig(
        f"SHAP_bar_{clean_target}.png",
        dpi=dpi
    )

    plt.savefig(
        f"SHAP_bar_{clean_target}.pdf"
    )

    plt.show()

In [ ]:
# ============================================================
# 10. SHAP analysis
# ============================================================

def compute_shap_for_rf(
    trained_pipeline,
    X_data,
    original_feature_names,
    feature_name_map=None
):
    """
    Computes SHAP values for a trained Random Forest pipeline.

    Works whether the pipeline contains:
    - imputer + feature_selector + model
    - imputer + model
    - model only
    """

    if feature_name_map is None:
        feature_name_map = {}

    X_working = X_data.copy()
    current_feature_names = np.array(original_feature_names)

    if "imputer" in trained_pipeline.named_steps:
        imputer = trained_pipeline.named_steps["imputer"]
        X_working = imputer.transform(X_working)
    else:
        X_working = X_working.values

    if "feature_selector" in trained_pipeline.named_steps:
        selector = trained_pipeline.named_steps["feature_selector"]
        X_working = selector.transform(X_working)
        selected_features = current_feature_names[selector.get_support()]
    else:
        selected_features = current_feature_names

    model = trained_pipeline.named_steps["model"]

    display_feature_names = [
        feature_name_map.get(f, f)
        for f in selected_features
    ]

    X_selected_df = pd.DataFrame(
        X_working,
        columns=display_feature_names,
        index=X_data.index
    )

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_selected_df)

    return explainer, shap_values, X_selected_df




## 4. Centralized geometry-conditioned sampling

All synthetic design-space samples used for geometry-conditioned explainability and Pareto exploration are generated here. This replaces the repeated sampling blocks in the working notebook and makes the workflow easier to audit.

The function fixes categorical/geometric variables such as number of stories and bay width, then samples the remaining numeric variables within the empirical quantile range of the corresponding subset. User-defined ranges can be supplied for design variables that require explicit control, such as average SCWB ratio or design drift.

In [ ]:
def filter_geometry_subset(X_original, fixed_geometry):
    """
    Return the subset of the original design matrix matching the requested geometry.

    Parameters
    ----------
    X_original : pandas.DataFrame
        Original feature matrix used to train the surrogate models.
    fixed_geometry : dict
        Geometry constraints, e.g., {"number of stories (NS)": 9, "bay width (BW), ft": 30}.
        Values may be scalars or (low, high) tuples.
    """
    X_filtered = X_original.copy()

    for col, value in fixed_geometry.items():
        if col not in X_original.columns:
            raise ValueError(f"'{col}' is not in X_original columns.")

        if isinstance(value, tuple):
            low, high = value
            X_filtered = X_filtered[(X_filtered[col] >= low) & (X_filtered[col] <= high)]
        else:
            X_filtered = X_filtered[X_filtered[col] == value]

    if X_filtered.empty:
        raise ValueError(f"No rows match fixed_geometry = {fixed_geometry}")

    return X_filtered


def make_geometry_conditioned_samples(
    X_original,
    fixed_geometry,
    custom_feature_ranges=None,
    n_samples=50000,
    random_state=RANDOM_STATE,
    range_quantiles=(0.05, 0.95),
):
    """
    Generate a reproducible synthetic design-space sample for one fixed geometry.

    This is the single source of synthetic sampling in the notebook. It is used by
    Sobol, ALE, tradeoff-map, and Pareto analyses to avoid inconsistent sampling
    assumptions across explainability methods.

    Returns
    -------
    X_sample : pandas.DataFrame
        Synthetic samples with the same column order as X_original.
    X_filtered : pandas.DataFrame
        Original rows matching the fixed geometry; used to define empirical ranges.
    """
    rng = np.random.default_rng(random_state)
    custom_feature_ranges = custom_feature_ranges or {}
    q_low, q_high = range_quantiles

    X_filtered = filter_geometry_subset(X_original, fixed_geometry)
    X_sample = pd.DataFrame(index=np.arange(n_samples))

    for col in X_original.columns:
        if col in fixed_geometry:
            value = fixed_geometry[col]
            X_sample[col] = X_filtered[col].median() if isinstance(value, tuple) else value

        elif col in custom_feature_ranges:
            low, high = custom_feature_ranges[col]
            X_sample[col] = rng.uniform(low, high, n_samples)

        elif np.issubdtype(X_original[col].dtype, np.number):
            low = X_filtered[col].quantile(q_low)
            high = X_filtered[col].quantile(q_high)
            X_sample[col] = low if np.isclose(low, high) else rng.uniform(low, high, n_samples)

        else:
            X_sample[col] = X_filtered[col].mode().iloc[0]

    return X_sample[X_original.columns], X_filtered


def generate_fixed_geometry_synthetic_X(
    X_original,
    fixed_geometry,
    custom_feature_ranges=None,
    n_random_samples=50000,
    random_state=RANDOM_STATE,
    range_quantiles=(0.05, 0.95),
):
    """Backward-compatible wrapper used by older analysis functions."""
    return make_geometry_conditioned_samples(
        X_original=X_original,
        fixed_geometry=fixed_geometry,
        custom_feature_ranges=custom_feature_ranges,
        n_samples=n_random_samples,
        random_state=random_state,
        range_quantiles=range_quantiles,
    )


def generate_geometry_fixed_random_X(
    X_original,
    fixed_geometry,
    n_samples=50000,
    random_state=RANDOM_STATE,
    range_quantiles=(0.05, 0.95),
    custom_feature_ranges=None,
):
    """Backward-compatible wrapper for ALE functions."""
    X_sample, _ = make_geometry_conditioned_samples(
        X_original=X_original,
        fixed_geometry=fixed_geometry,
        custom_feature_ranges=custom_feature_ranges,
        n_samples=n_samples,
        random_state=random_state,
        range_quantiles=range_quantiles,
    )
    return X_sample


def predict_rf_targets(X_sample, rf_models, target_cols=TARGET_COLS):
    """Append surrogate-model predictions for all target quantities."""
    pred_df = X_sample.copy()
    for target in target_cols:
        pred_df[target] = rf_models[target]["model"].predict(X_sample)
    return pred_df


# Default geometry-conditioned ranges used consistently across analyses.
GEOMETRY_CONDITIONED_RANGES = {
    "Avg_ColumnBeamRatio (AVG_SCWB)": (1.0, 3.0),
    "Avg_DgnStoryDrift": (0.001, 0.010),
}

# Backward-compatible variable name used in later example cells.
custom_feature_ranges = GEOMETRY_CONDITIONED_RANGES


## 5. Train surrogate models and evaluate predictive performance

In [ ]:
# ============================================================
# 6. Train RF and baseline Multiple Linear Regression models
# ============================================================
rf_models = {}
rf_results = []

for target in target_cols:
    y = data[target]

    y_train = y.loc[train_mask].copy()
    y_test = y.loc[test_mask].copy()

    # --------------------------------------------------------
    # Random Forest model
    # --------------------------------------------------------
    best_model, search, results_df, y_pred_train, y_pred_test = train_random_forest_for_target(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        target_name=target,
        random_state=42,
        n_iter=100
    )

    # --------------------------------------------------------
    # Baseline Multiple Linear Regression model
    # with RF-based feature selection
    # --------------------------------------------------------
    baseline_model, baseline_results_df, baseline_y_pred_train, baseline_y_pred_test = train_baseline_linear_regression(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        target_name=target,
        random_state=42
    )

    # --------------------------------------------------------
    # Store models, predictions, and outputs
    # --------------------------------------------------------
    rf_models[target] = {
        "model": best_model,
        "search": search,

        "y_train": y_train,
        "y_test": y_test,

        "y_pred_train": y_pred_train,
        "y_pred_test": y_pred_test,

        "baseline_model": baseline_model,
        "baseline_y_pred_train": baseline_y_pred_train,
        "baseline_y_pred_test": baseline_y_pred_test
    }

    # --------------------------------------------------------
    # Store performance results
    # --------------------------------------------------------
    rf_results.append(results_df)
    rf_results.append(baseline_results_df)

# ------------------------------------------------------------
# Combine summary results for RF and baseline models
# ------------------------------------------------------------
summary_results = pd.concat(rf_results, ignore_index=True)
summary_results

In [ ]:
# ============================================================
# 7. Plot predicted vs observed
# ============================================================
for target in target_cols:

    if target in ["Functional_DBE", "Functional_MCE"]:
        xlabel = "Observed FR (days)"
        ylabel = "Predicted FR (days)"
    elif target == "EAL":
        xlabel = "Observed EAL (USD)"
        ylabel = "Predicted EAL (USD)"
    else:
        xlabel = f"Observed {target}"
        ylabel = f"Predicted {target}"

    plot_predicted_vs_observed(
        y_train_true=rf_models[target]["y_train"],
        y_train_pred=rf_models[target]["y_pred_train"],

        y_test_true=rf_models[target]["y_test"],
        y_test_pred=rf_models[target]["y_pred_test"],

        xlabel=xlabel,
        ylabel=ylabel,
        target_name=target,
        title=f"{target}",

        train_color = "#0072B2",   # Okabe-Ito Blue
        test_color  = "#E69F00",   # Okabe-Ito Orange
        font_family="Times New Roman",
        figsize=(2.5, 2.5),


        save_path=f"RF_Predicted_vs_Observed_{target}.png"
    )


In [ ]:
for target in target_cols:        # Multiple Linear Regression baseline

    
    plot_predicted_vs_observed(
         y_train_true=rf_models[target]["y_train"],
         y_train_pred=rf_models[target]["baseline_y_pred_train"],
         y_test_true=rf_models[target]["y_test"],
         y_test_pred=rf_models[target]["baseline_y_pred_test"],
         xlabel="Observed FR (days)",
         ylabel="Predicted FR (days)",

         title=f"{target}",

         train_color = "#0072B2",   # Okabe-Ito Blue
         test_color  = "#E69F00",   # Okabe-Ito Orange

         train_alpha=0.35,
         test_alpha=0.70,

         train_marker_size=50,
         test_marker_size=65,

         font_family="Times New Roman",
         figsize=(2.5, 2.5),
         target_name=target,

        save_path=f"MLR_predicted_vs_observed_{target}.pdf"
        
     )

## 6. Global SHAP analysis on the held-out test set

This section evaluates feature contributions using the original held-out test cases. Geometry-conditioned explainability analyses are performed in later sections using the centralized sampling function.

In [ ]:
for target in target_cols:
    print("\nSHAP analysis for:", target)

    explainer, shap_values, X_selected_df = compute_shap_for_rf(
        trained_pipeline=rf_models[target]["model"],
        X_data=X_test,
        original_feature_names=X.columns,
        feature_name_map=feature_name_map,
    )

    plot_publication_shap(
        shap_values=shap_values,
        X_selected_df=X_selected_df,
        target=target,
        max_display=15,
        font_family="Times New Roman",
        font_size=14,
        dpi=300,
    )


## 7. Geometry-conditioned explainability and tradeoff analyses

The following functions use the centralized geometry-conditioned sample. This ensures the Sobol, ALE, tradeoff-map, and Pareto analyses are conditioned on the same sampling assumptions.

In [ ]:
# ============================================================
# True Sobol sensitivity analysis using trained RF surrogate
# Geometry-fixed analysis
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from SALib.sample import saltelli
from SALib.analyze import sobol


# ============================================================
# 1. Build Sobol problem for fixed geometry
# ============================================================

def build_sobol_problem_fixed_geometry(
    X_original,
    fixed_geometry,
    feature_cols=None,
    range_quantiles=(0.05, 0.95)
):
    """
    Builds SALib problem dictionary for true Sobol analysis.

    Geometry variables are fixed.
    All other numeric variables are allowed to vary within their observed
    conditional ranges for that fixed geometry.
    """

    # --------------------------------------------------------
    # Filter original database by fixed geometry
    # --------------------------------------------------------
    X_filtered = X_original.copy()

    for col, value in fixed_geometry.items():

        if col not in X_original.columns:
            raise ValueError(f"{col} is not in X_original columns.")

        if isinstance(value, tuple):
            lower, upper = value
            X_filtered = X_filtered[
                (X_filtered[col] >= lower) &
                (X_filtered[col] <= upper)
            ]

        else:
            X_filtered = X_filtered[
                X_filtered[col] == value
            ]

    if len(X_filtered) == 0:
        raise ValueError("No rows match the fixed geometry filters.")

    q_low, q_high = range_quantiles

    # --------------------------------------------------------
    # If user does not specify features, vary all non-fixed numeric features
    # --------------------------------------------------------
    if feature_cols is None:
        feature_cols = [
            col for col in X_original.columns
            if col not in fixed_geometry
            and np.issubdtype(X_original[col].dtype, np.number)
        ]

    # --------------------------------------------------------
    # Remove features with no meaningful range
    # --------------------------------------------------------
    sobol_features = []
    bounds = []

    for col in feature_cols:

        if col in fixed_geometry:
            continue

        if col not in X_original.columns:
            print(f"Skipping {col}: not in X_original.")
            continue

        if not np.issubdtype(X_original[col].dtype, np.number):
            print(f"Skipping {col}: not numeric.")
            continue

        low = X_filtered[col].quantile(q_low)
        high = X_filtered[col].quantile(q_high)

        if np.isclose(low, high):
            print(f"Skipping {col}: near-constant within fixed geometry.")
            continue

        sobol_features.append(col)
        bounds.append([low, high])

    problem = {
        "num_vars": len(sobol_features),
        "names": sobol_features,
        "bounds": bounds
    }

    print("Fixed geometry:")
    print(fixed_geometry)
    print("Matching rows:", len(X_filtered))
    print("Sobol variables:", len(sobol_features))

    return problem, X_filtered


# ============================================================
# 2. Generate Sobol samples and RF predictions
# ============================================================

def run_sobol_rf_fixed_geometry(
    X_original,
    rf_models,
    target_cols,
    fixed_geometry,
    feature_cols=None,
    base_sample_size=1024,
    range_quantiles=(0.05, 0.95),
    calc_second_order=False
):
    """
    Runs true Sobol sensitivity analysis using the trained RF surrogate.

    base_sample_size controls the Sobol sample size.
    Total model evaluations roughly:
        N * (D + 2) if calc_second_order=False
        N * (2D + 2) if calc_second_order=True
    """

    problem, X_filtered = build_sobol_problem_fixed_geometry(
        X_original=X_original,
        fixed_geometry=fixed_geometry,
        feature_cols=feature_cols,
        range_quantiles=range_quantiles
    )

    D = problem["num_vars"]

    if D == 0:
        raise ValueError("No varying variables available for Sobol analysis.")

    print("Base sample size:", base_sample_size)
    print("Number of variables:", D)

    # --------------------------------------------------------
    # Generate Saltelli/Sobol samples
    # --------------------------------------------------------
    param_values = saltelli.sample(
        problem,
        base_sample_size,
        calc_second_order=calc_second_order
    )

    print("Total RF evaluations:", param_values.shape[0])

    # --------------------------------------------------------
    # Build full X matrix for RF prediction
    # Start from median fixed-geometry row
    # --------------------------------------------------------
    base_row = X_filtered.median(numeric_only=True)

    # Enforce fixed geometry
    for col, value in fixed_geometry.items():
        if isinstance(value, tuple):
            base_row[col] = X_filtered[col].median()
        else:
            base_row[col] = value

    X_sobol = pd.DataFrame(
        np.repeat(
            base_row.values.reshape(1, -1),
            param_values.shape[0],
            axis=0
        ),
        columns=base_row.index
    )

    # Insert Sobol-sampled variables
    for i, col in enumerate(problem["names"]):
        X_sobol[col] = param_values[:, i]

    # Match RF input order exactly
    X_sobol = X_sobol[X_original.columns]

    # --------------------------------------------------------
    # Predict and analyze Sobol indices
    # --------------------------------------------------------
    all_results = []

    prediction_df = X_sobol.copy()

    for target in target_cols:

        print(f"Predicting and analyzing target: {target}")

        Y = rf_models[target]["model"].predict(X_sobol)

        prediction_df[target] = Y

        Si = sobol.analyze(
            problem,
            Y,
            calc_second_order=calc_second_order,
            print_to_console=False
        )

        result_df = pd.DataFrame({
            "Target": target,
            "Feature": problem["names"],
            "S1": Si["S1"],
            "S1_conf": Si["S1_conf"],
            "ST": Si["ST"],
            "ST_conf": Si["ST_conf"]
        })

        result_df = result_df.sort_values(
            "ST",
            ascending=False
        )

        all_results.append(result_df)

    sobol_results_df = pd.concat(
        all_results,
        ignore_index=True
    )

    return sobol_results_df, prediction_df, problem

In [ ]:
# ============================================================
# Plot Sobol sensitivity indices
# with excluded features and exactly 4 x-axis ticks
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

def plot_sobol_indices(
    sobol_results_df,
    target,
    top_n=12,
    feature_name_map=None,
    exclude_features=None,
    index_type="ST",
    title=None,
    save_path=None,
    figsize=(4, 4),
    dpi=300,
    font_family="Times New Roman",
    font_size=18
):
    plt.rcParams["font.family"] = font_family

    if feature_name_map is None:
        feature_name_map = {}

    if exclude_features is None:
        exclude_features = []

    if index_type not in ["S1", "ST"]:
        raise ValueError("index_type must be either 'S1' or 'ST'.")

    conf_col = f"{index_type}_conf"

    plot_df = sobol_results_df[
        sobol_results_df["Target"] == target
    ].copy()

    plot_df = plot_df[
        ~plot_df["Feature"].isin(exclude_features)
    ]

    plot_df = (
        plot_df
        .sort_values(index_type, ascending=False)
        .head(top_n)
        .copy()
    )

    plot_df["Feature_Display"] = plot_df["Feature"].map(
        lambda x: feature_name_map.get(x, x)
    )

    fig, ax = plt.subplots(
        figsize=figsize,
        dpi=dpi
    )

    ax.barh(
        plot_df["Feature_Display"][::-1],
        plot_df[index_type][::-1],
        xerr=plot_df[conf_col][::-1],
        color="#D9D9D9",
        edgecolor="black",
        linewidth=0.9,
        error_kw={
            "elinewidth": 0.9,
            "capsize": 3,
            "capthick": 0.9
        }
    )

    if index_type == "S1":
        xlabel = r"First-order Sobol index, $S_1$"
    else:
        xlabel = r"Total Sobol index, $S_T$"

    ax.set_xlabel(
        xlabel,
        fontsize=font_size
    )

    # --------------------------------------------------------
    # Force exactly 4 x-axis ticks with one decimal place
    # --------------------------------------------------------
    xmin, xmax = ax.get_xlim()

    if xmax <= 0:
        xmax = 1.0

    ax.set_xlim(0, xmax)

    ax.set_xticks(
        np.linspace(0, xmax, 4)
    )

    ax.xaxis.set_major_formatter(
        FormatStrFormatter("%.1f")
    )

    if title is not None:
        ax.set_title(
            title,
            fontsize=font_size + 1
        )

    ax.tick_params(
        axis="both",
        labelsize=font_size - 1,
        direction="in"
    )

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
        spine.set_color("black")

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight"
        )

        plt.savefig(
            save_path.replace(".png", ".pdf"),
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
# Uses centralized geometry-conditioned sampling functions defined above.

# ============================================================
# Fixed-geometry synthetic sampling + RF prediction + tradeoff plots
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import ScalarFormatter, MaxNLocator


# ------------------------------------------------------------
# 1. Generate fixed-geometry synthetic design samples
# ------------------------------------------------------------
def plot_tradeoff_fixed_geometry(
    df,
    x_col,
    y_col,
    color_col,
    xlabel=None,
    ylabel=None,
    cbar_label=None,
    save_path=None,
    figsize=(3.5, 3.2),
    dpi=300,
    font_family="Times New Roman",
    font_size=16,
    point_size=28,
    point_alpha=0.78,
    edge_color="black",
    edge_width=0.25,
    show_grid=False,
    cmap=None
):

    plt.rcParams["font.family"] = font_family

    if xlabel is None:
        xlabel = x_col

    if ylabel is None:
        ylabel = y_col

    if cbar_label is None:
        cbar_label = color_col

    if cmap is None:
        cmap = LinearSegmentedColormap.from_list(
            "blue_coral",
            [
                "#3B4CC0",
                "#7B8AE6",
                "#C6B7E6",
                "#F4A582",
                "#EE6A50"
            ]
        )

    fig, ax = plt.subplots(
        figsize=figsize,
        dpi=dpi
    )

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    sc = ax.scatter(
        df[x_col],
        df[y_col],
        c=df[color_col],
        cmap=cmap,
        s=point_size,
        alpha=point_alpha,
        edgecolors=edge_color,
        linewidths=edge_width
    )

    ax.set_xlabel(
        xlabel,
        fontsize=font_size
    )

    ax.set_ylabel(
        ylabel,
        fontsize=font_size
    )

    # --------------------------------------------------
    # Nice publication-quality ticks (4–5 major ticks)
    # --------------------------------------------------
    ax.xaxis.set_major_locator(
        MaxNLocator(
            nbins=4,
            min_n_ticks=4
        )
    )

    ax.yaxis.set_major_locator(
        MaxNLocator(
            nbins=4,
            min_n_ticks=4
        )
    )

    # Scientific notation only for EAL axis
    if y_col == "EAL":

        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((0, 0))
        formatter.set_useOffset(False)

        ax.yaxis.set_major_formatter(formatter)

    ax.tick_params(
        axis="both",
        labelsize=font_size - 1,
        direction="in",
        length=5,
        width=1.0
    )

    ax.grid(show_grid)

    # Publication border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)
        spine.set_color("black")

    # Colorbar
    cbar = fig.colorbar(
        sc,
        ax=ax,
        pad=0.025,
        aspect=25
    )

    cbar.set_label(
        cbar_label,
        fontsize=font_size
    )

    cbar.ax.tick_params(
        labelsize=font_size - 1,
        direction="in",
        length=4,
        width=1.0
    )

    cbar.outline.set_linewidth(1.0)

    plt.tight_layout()

    if save_path is not None:

        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight"
        )

        plt.savefig(
            save_path.replace(".png", ".pdf"),
            bbox_inches="tight"
        )

    plt.show()

# ------------------------------------------------------------
# 4. Full workflow: generate synthetic data + predict + plot
# ------------------------------------------------------------
def run_fixed_geometry_tradeoff_analysis(
    X_original,
    rf_models,
    fixed_geometry,
    color_col,
    cbar_label=None,
    custom_feature_ranges=None,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    save_prefix="FixedGeometry_Tradeoff"
):
    X_syn, X_filtered = generate_fixed_geometry_synthetic_X(
        X_original=X_original,
        fixed_geometry=fixed_geometry,
        custom_feature_ranges=custom_feature_ranges,
        n_random_samples=n_random_samples,
        random_state=random_state,
        range_quantiles=range_quantiles
    )

    tradeoff_df = predict_rf_targets(
        X_syn=X_syn,
        rf_models=rf_models,
        target_cols=("Functional_DBE", "Functional_MCE", "EAL")
    )

    print("\nColor variable range:")
    print(tradeoff_df[color_col].describe())

    plot_tradeoff_fixed_geometry(
        df=tradeoff_df,
        x_col="Functional_DBE",
        y_col="Functional_MCE",
        color_col=color_col,
        xlabel="DBE recovery time (days)",
        ylabel="MCE recovery time (days)",
        cbar_label=cbar_label,
        save_path=f"{save_prefix}_DBE_MCE.png"
    )

    plot_tradeoff_fixed_geometry(
        df=tradeoff_df,
        x_col="Functional_MCE",
        y_col="EAL",
        color_col=color_col,
        xlabel="MCE recovery time (days)",
        ylabel="EAL ($)",
        cbar_label=cbar_label,
        save_path=f"{save_prefix}_MCE_EAL.png"
    )

    plot_tradeoff_fixed_geometry(
        df=tradeoff_df,
        x_col="Functional_DBE",
        y_col="EAL",
        color_col=color_col,
        xlabel="DBE recovery time (days)",
        ylabel="EAL ($)",
        cbar_label=cbar_label,
        save_path=f"{save_prefix}_DBE_EAL.png"
    )

    return tradeoff_df, X_syn, X_filtered

In [ ]:
# Uses centralized geometry-conditioned sampling functions defined above.

# ============================================================
# Fixed-geometry synthetic sampling + combined ALE
# optional custom ranges + no legend + smart scientific notation
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from matplotlib.ticker import MaxNLocator, ScalarFormatter


def compute_ale_1d(model, X_data, feature, grid_size=35):
    X_work = X_data.copy()
    x = X_work[feature].values

    quantiles = np.unique(
        np.quantile(
            x,
            np.linspace(0, 1, grid_size + 1)
        )
    )

    if len(quantiles) < 3:
        raise ValueError(f"{feature} has too few unique values for ALE.")

    bin_ids = np.digitize(
        x,
        quantiles[1:-1],
        right=True
    )

    effects = []
    centers = []

    for k in range(len(quantiles) - 1):

        idx = bin_ids == k

        if idx.sum() == 0:
            continue

        X_low = X_work.loc[idx].copy()
        X_high = X_work.loc[idx].copy()

        X_low[feature] = quantiles[k]
        X_high[feature] = quantiles[k + 1]

        effects.append(
            np.mean(model.predict(X_high) - model.predict(X_low))
        )

        centers.append(
            0.5 * (quantiles[k] + quantiles[k + 1])
        )

    ale_values = np.cumsum(effects)
    ale_values = ale_values - np.mean(ale_values)

    return pd.DataFrame({
        "Feature_Value": centers,
        "ALE": ale_values
    })


def normalize_ale_curve(ale_values):
    ale_values = np.asarray(ale_values)
    ale_range = np.max(ale_values) - np.min(ale_values)

    if np.isclose(ale_range, 0):
        return np.zeros_like(ale_values)

    return ale_values / ale_range


def apply_publication_axis_format(ax):
    """
    Uses 4–5 major ticks and switches to scientific notation
    for values outside approximately 10^-1 to 10^2.
    """

    ax.xaxis.set_major_locator(
        MaxNLocator(nbins=5, min_n_ticks=4)
    )

    ax.yaxis.set_major_locator(
        MaxNLocator(nbins=5, min_n_ticks=4)
    )

    formatter_x = ScalarFormatter(useMathText=True)
    formatter_x.set_powerlimits((-1, 2))
    formatter_x.set_useOffset(False)

    formatter_y = ScalarFormatter(useMathText=True)
    formatter_y.set_powerlimits((-1, 2))
    formatter_y.set_useOffset(False)

    ax.xaxis.set_major_formatter(formatter_x)
    ax.yaxis.set_major_formatter(formatter_y)


def plot_combined_ale_for_feature(
    X_data,
    rf_models,
    feature,
    target_cols,
    feature_name_map=None,
    target_label_map=None,
    grid_size=35,
    smooth=True,
    smooth_sigma=1.0,
    save_path=None,
    show_legend=False,
    figsize=(3.5, 3.0),
    dpi=300,
    font_family="Times New Roman",
    font_size=16
):
    plt.rcParams["font.family"] = font_family

    if feature_name_map is None:
        feature_name_map = {}

    if target_label_map is None:
        target_label_map = {
            "Functional_DBE": "DBE recovery",
            "Functional_MCE": "MCE recovery",
            "EAL": "EAL"
        }

    feature_label = feature_name_map.get(feature, feature)

    target_style_map = {
        "Functional_DBE": {
            "color": "blue",
            "linestyle": "-",
            "linewidth": 2.4
        },
        "Functional_MCE": {
            "color": "red",
            "linestyle": "--",
            "linewidth": 2.4
        },
        "EAL": {
            "color": "black",
            "linestyle": "-.",
            "linewidth": 2.4
        }
    }

    fig, ax = plt.subplots(
        figsize=figsize,
        dpi=dpi
    )

    for target in target_cols:

        ale_df = compute_ale_1d(
            model=rf_models[target]["model"],
            X_data=X_data,
            feature=feature,
            grid_size=grid_size
        )

        ale_norm = normalize_ale_curve(
            ale_df["ALE"].values
        )

        if smooth:
            ale_norm = gaussian_filter1d(
                ale_norm,
                sigma=smooth_sigma
            )

        style = target_style_map.get(
            target,
            {
                "color": "black",
                "linestyle": "-",
                "linewidth": 2.4
            }
        )

        ax.plot(
            ale_df["Feature_Value"],
            ale_norm,
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=style["linewidth"],
            label=target_label_map.get(target, target)
        )

    ax.axhline(
        0,
        color="gray",
        linestyle=":",
        linewidth=1.0
    )

    ax.set_xlabel(
        feature_label,
        fontsize=font_size
    )

    ax.set_ylabel(
        "Normalized ALE",
        fontsize=font_size
    )

    apply_publication_axis_format(ax)

    ax.tick_params(
        axis="both",
        labelsize=font_size - 1,
        direction="in",
        length=5,
        width=1.0
    )

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
        spine.set_color("black")

    if show_legend:
        ax.legend(
            frameon=False,
            fontsize=font_size - 1,
            loc="best"
        )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight"
        )

        plt.savefig(
            save_path.replace(".png", ".pdf"),
            bbox_inches="tight"
        )

    plt.show()


def run_fixed_geometry_combined_ale(
    X_original,
    rf_models,
    fixed_geometry,
    selected_features,
    target_cols=["Functional_DBE", "Functional_MCE", "EAL"],
    custom_feature_ranges=None,
    feature_name_map=None,
    target_label_map=None,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    grid_size=35,
    smooth=True,
    smooth_sigma=1.0,
    show_legend=False,
    save_prefix="Combined_ALE"
):
    X_ale, X_filtered = generate_geometry_fixed_random_X(
        X_original=X_original,
        fixed_geometry=fixed_geometry,
        selected_features=selected_features,
        custom_feature_ranges=custom_feature_ranges,
        n_random_samples=n_random_samples,
        random_state=random_state,
        range_quantiles=range_quantiles
    )

    for feature in selected_features:

        if feature not in X_ale.columns:
            print(f"Skipping {feature}: not found.")
            continue

        clean_feature = (
            feature.replace(" ", "_")
                   .replace("(", "")
                   .replace(")", "")
                   .replace("/", "_")
        )

        plot_combined_ale_for_feature(
            X_data=X_ale,
            rf_models=rf_models,
            feature=feature,
            target_cols=target_cols,
            feature_name_map=feature_name_map,
            target_label_map=target_label_map,
            grid_size=grid_size,
            smooth=smooth,
            smooth_sigma=smooth_sigma,
            show_legend=show_legend,
            save_path=f"{save_prefix}_{clean_feature}.png"
        )

    return X_ale, X_filtered

In [ ]:
# ============================================================
# IDEA 4: Tradeoff maps for top Sobol variables
# ============================================================

from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import ScalarFormatter


def plot_tradeoff_map(
    df,
    x_col,
    y_col,
    color_col,
    xlabel,
    ylabel,
    cbar_label=None,
    title=None,
    save_path=None,
    figsize=(6.4, 5.2),
    dpi=300,
    font_family="Times New Roman",
    font_size=12,
    point_size=28,
    point_alpha=0.78,
    edge_color="black",
    edge_width=0.25
):
    plt.rcParams["font.family"] = font_family

    if cbar_label is None:
        cbar_label = color_col

    blue_coral = LinearSegmentedColormap.from_list(
        "blue_coral",
        [
            "#3B4CC0",
            "#7B8AE6",
            "#C6B7E6",
            "#F4A582",
            "#EE6A50"
        ]
    )

    plot_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    norm = Normalize(
        vmin=plot_df[color_col].min(),
        vmax=plot_df[color_col].max()
    )

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    sc = ax.scatter(
        plot_df[x_col],
        plot_df[y_col],
        c=plot_df[color_col],
        cmap=blue_coral,
        norm=norm,
        s=point_size,
        alpha=point_alpha,
        edgecolors=edge_color,
        linewidths=edge_width
    )

    ax.set_xlabel(xlabel, fontsize=font_size)
    ax.set_ylabel(ylabel, fontsize=font_size)

    if title is not None:
        ax.set_title(title, fontsize=font_size + 1)

    if y_col == "EAL":
        formatter_y = ScalarFormatter(useMathText=True)
        formatter_y.set_powerlimits((0, 0))
        formatter_y.set_useOffset(False)
        ax.yaxis.set_major_formatter(formatter_y)

    ax.tick_params(
        axis="both",
        labelsize=font_size - 1,
        direction="in"
    )

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)
        spine.set_color("black")

    cbar = fig.colorbar(sc, ax=ax, pad=0.025, aspect=25)
    cbar.set_label(cbar_label, fontsize=font_size)
    cbar.ax.tick_params(labelsize=font_size - 1, direction="in")
    cbar.outline.set_linewidth(1.0)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")

    plt.show()


def run_tradeoff_maps_for_top_sobol_features(
    sobol_results_df,
    sobol_prediction_df,
    target_for_ranking="EAL",
    top_n=5,
    index_type="ST",
    exclude_features=None,
    feature_name_map=None,
    save_prefix="Tradeoff"
):
    if exclude_features is None:
        exclude_features = []

    if feature_name_map is None:
        feature_name_map = {}

    top_features = (
        sobol_results_df[
            (sobol_results_df["Target"] == target_for_ranking) &
            (~sobol_results_df["Feature"].isin(exclude_features))
        ]
        .sort_values(index_type, ascending=False)
        .head(top_n)["Feature"]
        .tolist()
    )

    print(f"Top {top_n} features based on {target_for_ranking}:")
    print(top_features)

    for feature in top_features:

        cbar_label = feature_name_map.get(feature, feature)

        clean_feature = (
            feature.replace(" ", "_")
                   .replace("(", "")
                   .replace(")", "")
                   .replace("/", "_")
        )

        plot_tradeoff_map(
            df=sobol_prediction_df,
            x_col="Functional_DBE",
            y_col="EAL",
            color_col=feature,
            xlabel="DBE recovery time (days)",
            ylabel="EAL ($)",
            cbar_label=cbar_label,
            title=None,
            save_path=f"{save_prefix}_{clean_feature}_DBE_EAL.png"
        )

        plot_tradeoff_map(
            df=sobol_prediction_df,
            x_col="Functional_MCE",
            y_col="EAL",
            color_col=feature,
            xlabel="MCE recovery time (days)",
            ylabel="EAL ($)",
            cbar_label=cbar_label,
            title=None,
            save_path=f"{save_prefix}_{clean_feature}_MCE_EAL.png"
        )

        plot_tradeoff_map(
            df=sobol_prediction_df,
            x_col="Functional_DBE",
            y_col="Functional_MCE",
            color_col=feature,
            xlabel="DBE recovery time (days)",
            ylabel="MCE recovery time (days)",
            cbar_label=cbar_label,
            title=None,
            save_path=f"{save_prefix}_{clean_feature}_DBE_MCE.png"
        )

In [ ]:
# Uses centralized geometry-conditioned sampling functions defined above.

# ============================================================
# Fixed-geometry synthetic sampling + RF prediction
# + Pareto front + clean 3D visualization
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, MaxNLocator


# ------------------------------------------------------------
# 1. Generate fixed-geometry synthetic samples
# ------------------------------------------------------------
def identify_pareto_minimization(df, objective_cols):
    values = df[objective_cols].to_numpy()
    n = values.shape[0]

    is_pareto = np.ones(n, dtype=bool)

    for i in range(n):
        if is_pareto[i]:
            dominated_by_i = (
                np.all(values[i] <= values, axis=1)
                &
                np.any(values[i] < values, axis=1)
            )
            is_pareto[dominated_by_i] = False

    return is_pareto


def add_pareto_information(
    df,
    objective_cols=("Functional_DBE", "Functional_MCE", "EAL")
):
    out = df.copy()

    out["Pareto_Efficient"] = identify_pareto_minimization(
        out,
        objective_cols=list(objective_cols)
    )

    norm_obj = out[list(objective_cols)].copy()

    for col in objective_cols:
        col_min = norm_obj[col].min()
        col_max = norm_obj[col].max()

        if np.isclose(col_max, col_min):
            norm_obj[col] = 0.0
        else:
            norm_obj[col] = (norm_obj[col] - col_min) / (col_max - col_min)

    out["Distance_to_Ideal"] = np.sqrt(
        np.sum(norm_obj.to_numpy() ** 2, axis=1)
    )

    out["Compromise_Rank"] = out["Distance_to_Ideal"].rank(
        method="first",
        ascending=True
    )

    print("Total designs:", len(out))
    print("Pareto-optimal designs:", out["Pareto_Efficient"].sum())
    print("Pareto percentage:", 100 * out["Pareto_Efficient"].mean())

    return out


# ------------------------------------------------------------
# 4. Clean 3D Pareto plot
# ------------------------------------------------------------
def plot_3d_pareto_front_fixed_geometry(
    df,
    x_col="Functional_DBE",
    y_col="Functional_MCE",
    z_col="EAL",
    pareto_col="Pareto_Efficient",
    xlabel="DBE recovery time (days)",
    ylabel="MCE recovery time (days)",
    zlabel="EAL ($)",
    save_path=None,
    figsize=(6.5, 6.2),
    dpi=300,
    font_family="Times New Roman",
    font_size=18,
    background_point_size=12,
    pareto_point_size=58,
    background_alpha=0.35,
    pareto_alpha=1.0,
    background_edge="#9A9A9A",
    pareto_color="#EE6A50",
    pareto_edge="black",
    background_linewidth=0.35,
    pareto_linewidth=0.50,
    elev=24,
    azim=-55,
    show_legend=False
):
    plt.rcParams["font.family"] = font_family

    fig = plt.figure(figsize=figsize, dpi=dpi)
    ax = fig.add_subplot(111, projection="3d")

    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # --------------------------------------------------------
    # Clean default 3D panes/grid
    # No manual cube edges are drawn anywhere in this function.
    # --------------------------------------------------------
    ax.grid(False)

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.yaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.zaxis.pane.set_edgecolor((1, 1, 1, 0))

    # Clean white panes, but keep axis edges visible
    ax.grid(False)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
     axis.pane.fill = False
     axis.pane.set_edgecolor("black")
     axis.pane.set_linewidth(2.0)

    # Keep main axis lines visible
    ax.xaxis.line.set_color("black")
    ax.yaxis.line.set_color("black")
    ax.zaxis.line.set_color("black")

    ax.xaxis.line.set_linewidth(2.0)
    ax.yaxis.line.set_linewidth(2.0)
    ax.zaxis.line.set_linewidth(2.0)

    non_pareto = ~df[pareto_col]
    pareto = df[pareto_col]

    # --------------------------------------------------------
    # Background designs: hollow grey circles
    # --------------------------------------------------------
    ax.scatter(
        df.loc[non_pareto, x_col],
        df.loc[non_pareto, y_col],
        df.loc[non_pareto, z_col],
        facecolors="none",
        edgecolors=background_edge,
        s=background_point_size,
        alpha=background_alpha,
        linewidths=background_linewidth,
        depthshade=False,
        marker="o",
        label="Dominated designs"
    )

    # --------------------------------------------------------
    # Pareto-optimal designs: filled coral circles
    # --------------------------------------------------------
    ax.scatter(
        df.loc[pareto, x_col],
        df.loc[pareto, y_col],
        df.loc[pareto, z_col],
        color=pareto_color,
        edgecolors=pareto_edge,
        s=pareto_point_size,
        alpha=pareto_alpha,
        linewidths=pareto_linewidth,
        depthshade=False,
        marker="o",
        label="Pareto-optimal designs"
    )

    ax.set_xlabel(xlabel, fontsize=font_size, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=font_size, labelpad=10)
    ax.set_zlabel(zlabel, fontsize=font_size, labelpad=14)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=4))
    ax.zaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=4))

    formatter_x = ScalarFormatter(useMathText=True)
    formatter_x.set_powerlimits((-1, 2))
    formatter_x.set_useOffset(False)

    formatter_y = ScalarFormatter(useMathText=True)
    formatter_y.set_powerlimits((-1, 2))
    formatter_y.set_useOffset(False)

    formatter_z = ScalarFormatter(useMathText=True)
    formatter_z.set_powerlimits((-1, 2))
    formatter_z.set_useOffset(False)

    ax.xaxis.set_major_formatter(formatter_x)
    ax.yaxis.set_major_formatter(formatter_y)
    ax.zaxis.set_major_formatter(formatter_z)

    ax.tick_params(
        axis="both",
        labelsize=font_size - 1,
        pad=3
    )

    ax.tick_params(
        axis="z",
        labelsize=font_size - 1,
        pad=3
    )

    ax.view_init(elev=elev, azim=azim)

    try:
        ax.set_box_aspect((1.1, 1.1, 0.85))
    except Exception:
        pass

    if show_legend:
        ax.legend(
            frameon=False,
            fontsize=font_size - 1,
            loc="upper left"
        )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")

    plt.show()


# ------------------------------------------------------------
# 5. Full workflow
# ------------------------------------------------------------
def run_fixed_geometry_pareto_3d(
    X_original,
    rf_models,
    fixed_geometry,
    custom_feature_ranges=None,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    objective_cols=("Functional_DBE", "Functional_MCE", "EAL"),
    save_prefix="FixedGeometry_Pareto",
    show_legend=False,
    elev=24,
    azim=-55
):
    X_syn, X_filtered = generate_fixed_geometry_synthetic_X(
        X_original=X_original,
        fixed_geometry=fixed_geometry,
        custom_feature_ranges=custom_feature_ranges,
        n_random_samples=n_random_samples,
        random_state=random_state,
        range_quantiles=range_quantiles
    )

    pred_df = predict_rf_targets(
        X_syn=X_syn,
        rf_models=rf_models,
        target_cols=objective_cols
    )

    pareto_df = add_pareto_information(
        df=pred_df,
        objective_cols=objective_cols
    )

    plot_3d_pareto_front_fixed_geometry(
        df=pareto_df,
        x_col="Functional_DBE",
        y_col="Functional_MCE",
        z_col="EAL",
        xlabel="DBE recovery time (days)",
        ylabel="MCE recovery time (days)",
        zlabel="EAL ($)",
        save_path=f"{save_prefix}_3D.png",
        show_legend=show_legend,
        elev=elev,
        azim=azim
    )

    return pareto_df, X_syn, X_filtered

In [ ]:
# ============================================================
# Pareto vs non-Pareto comparison plots
# Boxplots + feature enrichment only
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter


# ------------------------------------------------------------
# Helper formatting
# ------------------------------------------------------------
def apply_publication_format(
    ax,
    font_size=18,
    axis="y"
):
    ax.tick_params(
        axis="both",
        labelsize=font_size - 2,
        direction="in",
        length=6,
        width=1.2
    )

    if axis in ["y", "both"]:
        ax.yaxis.set_major_locator(
            MaxNLocator(nbins=5, min_n_ticks=4)
        )

        formatter_y = ScalarFormatter(useMathText=True)
        formatter_y.set_powerlimits((-1, 2))
        formatter_y.set_useOffset(False)
        ax.yaxis.set_major_formatter(formatter_y)

    if axis in ["x", "both"]:
        ax.xaxis.set_major_locator(
            MaxNLocator(nbins=5, min_n_ticks=4)
        )

        formatter_x = ScalarFormatter(useMathText=True)
        formatter_x.set_powerlimits((-1, 2))
        formatter_x.set_useOffset(False)
        ax.xaxis.set_major_formatter(formatter_x)

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.3)
        spine.set_color("black")


# ============================================================
# 1. Boxplot: Pareto vs non-Pareto distributions
# ============================================================
def plot_pareto_vs_nonpareto_boxplot(
    df,
    feature_cols,
    feature_label_map=None,
    pareto_col="Pareto_Efficient",
    standardize=True,
    save_path=None,
    figsize=(5, 3),
    dpi=300,
    font_family="Times New Roman",
    font_size=18,
    nonpareto_color="#D0D0D0",
    pareto_color="#EE6A50"
):
    """
    Creates paired boxplots comparing non-Pareto and Pareto-optimal designs.

    If standardize=True, each feature is transformed as:
        z = (x - mean_all) / std_all

    This allows SCWB, IDR, WL, and T1 to be compared on one axis.
    """

    plt.rcParams["font.family"] = font_family

    if feature_label_map is None:
        feature_label_map = {}

    data = []
    positions = []
    colors = []

    xtick_positions = []
    xtick_labels = []

    pos = 1.0

    for feature in feature_cols:

        all_vals = df[feature].dropna()
        non_vals = df.loc[~df[pareto_col], feature].dropna().values
        par_vals = df.loc[df[pareto_col], feature].dropna().values

        if standardize:
            mu = all_vals.mean()
            sig = all_vals.std()

            if np.isclose(sig, 0):
                non_vals = np.zeros_like(non_vals)
                par_vals = np.zeros_like(par_vals)
            else:
                non_vals = (non_vals - mu) / sig
                par_vals = (par_vals - mu) / sig

        data.extend([non_vals, par_vals])
        positions.extend([pos, pos + 0.42])
        colors.extend([nonpareto_color, pareto_color])

        xtick_positions.append(pos + 0.21)
        xtick_labels.append(feature_label_map.get(feature, feature))

        pos += 1.35

    fig, ax = plt.subplots(
        figsize=figsize,
        dpi=dpi
    )

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.32,
        patch_artist=True,
        showfliers=True,
        whis=1.5,
        boxprops=dict(
            linewidth=1.2,
            edgecolor="black"
        ),
        medianprops=dict(
            linewidth=2.0,
            color="black"
        ),
        whiskerprops=dict(
            linewidth=1.2,
            color="black"
        ),
        capprops=dict(
            linewidth=1.2,
            color="black"
        ),
        flierprops=dict(
            marker="o",
            markersize=2.8,
            markerfacecolor="black",
            markeredgecolor="black",
            alpha=0.45
        )
    )

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.90)

    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(
        xtick_labels,
        fontsize=font_size
    )

    if standardize:
        ax.set_ylabel(
            "Standardized feature value",
            fontsize=font_size
        )
        ax.axhline(
            0.0,
            color="black",
            linestyle=":",
            linewidth=1.1
        )
    else:
        ax.set_ylabel(
            "Feature value",
            fontsize=font_size
        )

    apply_publication_format(
        ax,
        font_size=font_size,
        axis="y"
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight"
        )

        plt.savefig(
            save_path.replace(".png", ".pdf"),
            bbox_inches="tight"
        )

    plt.show()


# ============================================================
# 2. Feature enrichment calculation
# ============================================================
def compute_pareto_feature_enrichment(
    df,
    feature_cols,
    pareto_col="Pareto_Efficient"
):
    """
    Computes how Pareto-optimal designs differ from non-Pareto designs.

    Percent change:
        100 * (mean_Pareto - mean_NonPareto) / mean_NonPareto

    Effect size:
        (mean_Pareto - mean_NonPareto) / std_All
    """

    rows = []

    for feature in feature_cols:

        all_vals = df[feature].dropna()
        non_vals = df.loc[~df[pareto_col], feature].dropna()
        par_vals = df.loc[df[pareto_col], feature].dropna()

        if len(non_vals) == 0 or len(par_vals) == 0:
            continue

        non_mean = non_vals.mean()
        par_mean = par_vals.mean()
        std_all = all_vals.std()

        if np.isclose(non_mean, 0):
            percent_change = np.nan
        else:
            percent_change = 100.0 * (par_mean - non_mean) / non_mean

        if np.isclose(std_all, 0):
            effect_size = np.nan
        else:
            effect_size = (par_mean - non_mean) / std_all

        rows.append({
            "Feature": feature,
            "NonPareto_Mean": non_mean,
            "Pareto_Mean": par_mean,
            "Percent_Change": percent_change,
            "Effect_Size": effect_size
        })

    return pd.DataFrame(rows)


# ============================================================
# 3. Feature enrichment bar plot
# ============================================================
def plot_pareto_feature_enrichment(
    df,
    feature_cols,
    feature_label_map=None,
    pareto_col="Pareto_Efficient",
    metric="Percent_Change",
    save_path=None,
    figsize=(5, 3),
    dpi=300,
    font_family="Times New Roman",
    font_size=18,
    positive_color="#EE6A50",
    negative_color="#D0D0D0"
):
    """
    Plots feature enrichment for Pareto-optimal designs.

    metric options:
        "Percent_Change" = percent change in Pareto mean relative to non-Pareto mean
        "Effect_Size"    = standardized mean difference
    """

    plt.rcParams["font.family"] = font_family

    if feature_label_map is None:
        feature_label_map = {}

    enrich_df = compute_pareto_feature_enrichment(
        df=df,
        feature_cols=feature_cols,
        pareto_col=pareto_col
    )

    if metric not in enrich_df.columns:
        raise ValueError(
            f"metric must be one of {list(enrich_df.columns)}"
        )

    enrich_df["Feature_Display"] = enrich_df["Feature"].map(
        lambda x: feature_label_map.get(x, x)
    )

    enrich_df = enrich_df.sort_values(
        metric,
        ascending=True
    )

    colors = ["#EE6A50"] * len(enrich_df)

    fig, ax = plt.subplots(
        figsize=figsize,
        dpi=dpi
    )

    ax.barh(
        enrich_df["Feature_Display"],
        enrich_df[metric],
        color=colors,
        edgecolor="black",
        linewidth=1.1
    )

    ax.axvline(
        0,
        color="black",
        linewidth=1.2
    )

    if metric == "Percent_Change":
        ax.set_xlabel(
            "Change in Pareto mean relative to non-Pareto mean (%)",
            fontsize=font_size
        )
    elif metric == "Effect_Size":
        ax.set_xlabel(
            r"Standardized difference, $(\mu_{\mathrm{Pareto}}-\mu_{\mathrm{NonPareto}})/\sigma_{\mathrm{All}}$",
            fontsize=font_size
        )
    else:
        ax.set_xlabel(
            metric,
            fontsize=font_size
        )

    apply_publication_format(
        ax,
        font_size=font_size,
        axis="x"
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight"
        )

        plt.savefig(
            save_path.replace(".png", ".pdf"),
            bbox_inches="tight"
        )

    plt.show()

    return enrich_df

In [ ]:
# Uses centralized geometry-conditioned sampling functions defined above.

# ============================================================
# Aggregate Pareto fronts across fixed geometries
# + train interpretable classifiers
# + visualize decision rules and feature importance
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
from matplotlib.ticker import MaxNLocator, ScalarFormatter


# ============================================================
# 1. Synthetic sampling for one fixed geometry
# ============================================================

# ============================================================
# 2. Predict DBE, MCE, EAL using trained RF surrogates
# ============================================================

# ============================================================
# 3. Fast Pareto identification for 3 minimization objectives
# ============================================================

def identify_pareto_3obj_min(df, objective_cols):
    """
    Efficient Pareto identification for three minimization objectives.

    A point is Pareto-optimal if no other point is equal/better
    in all objectives and strictly better in at least one objective.
    """

    values = df[objective_cols].to_numpy()
    n = values.shape[0]

    order = np.lexsort((values[:, 2], values[:, 1], values[:, 0]))
    is_pareto_sorted = np.zeros(n, dtype=bool)

    front = []

    for idx_sorted, idx_original in enumerate(order):

        x, y, z = values[idx_original]

        dominated = False

        for f_idx in front:
            fy, fz = values[f_idx, 1], values[f_idx, 2]

            if fy <= y and fz <= z:
                dominated = True
                break

        if not dominated:
            is_pareto_sorted[idx_sorted] = True

            # remove front points dominated by current point in y-z space
            new_front = []
            for f_idx in front:
                fy, fz = values[f_idx, 1], values[f_idx, 2]
                if not (y <= fy and z <= fz):
                    new_front.append(f_idx)

            new_front.append(idx_original)
            front = new_front

    is_pareto = np.zeros(n, dtype=bool)
    is_pareto[order] = is_pareto_sorted

    return is_pareto


def add_pareto_information(
    df,
    objective_cols=("Functional_DBE", "Functional_MCE", "EAL")
):
    out = df.copy()

    out["Pareto_Efficient"] = identify_pareto_3obj_min(
        out,
        objective_cols=list(objective_cols)
    )

    norm_obj = out[list(objective_cols)].copy()

    for col in objective_cols:
        cmin = norm_obj[col].min()
        cmax = norm_obj[col].max()

        if np.isclose(cmax, cmin):
            norm_obj[col] = 0.0
        else:
            norm_obj[col] = (norm_obj[col] - cmin) / (cmax - cmin)

    out["Distance_to_Ideal"] = np.sqrt(
        np.sum(norm_obj.to_numpy() ** 2, axis=1)
    )

    out["Compromise_Rank"] = out["Distance_to_Ideal"].rank(
        method="first",
        ascending=True
    )

    return out


# ============================================================
# 4. Generate Pareto data across multiple fixed geometries
# ============================================================

def build_multi_geometry_pareto_dataset(
    X_original,
    rf_models,
    building_cases,
    custom_feature_ranges=None,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    objective_cols=("Functional_DBE", "Functional_MCE", "EAL")
):
    all_results = []

    for i, (case_name, fixed_geometry) in enumerate(building_cases.items()):

        print("\n" + "=" * 70)
        print(f"Running geometry: {case_name}")
        print("=" * 70)

        X_syn, X_filtered = generate_fixed_geometry_synthetic_X(
            X_original=X_original,
            fixed_geometry=fixed_geometry,
            custom_feature_ranges=custom_feature_ranges,
            n_random_samples=n_random_samples,
            random_state=random_state + i,
            range_quantiles=range_quantiles
        )

        pred_df = predict_rf_targets(
            X_syn=X_syn,
            rf_models=rf_models,
            target_cols=objective_cols
        )

        pred_df = add_pareto_information(
            df=pred_df,
            objective_cols=objective_cols
        )

        pred_df["Geometry_Case"] = case_name

        for key, value in fixed_geometry.items():
            pred_df[f"Fixed_{key}"] = value

        print("Total synthetic designs:", len(pred_df))
        print("Pareto designs:", pred_df["Pareto_Efficient"].sum())
        print("Pareto percentage:", 100 * pred_df["Pareto_Efficient"].mean())

        all_results.append(pred_df)

    all_pareto_df = pd.concat(
        all_results,
        ignore_index=True
    )

    return all_pareto_df


# ============================================================
# 5. Balance dataset for classifier training
# ============================================================

def create_balanced_pareto_classifier_dataset(
    df,
    samples_per_class_per_geometry=100,
    pareto_col="Pareto_Efficient",
    geometry_col="Geometry_Case",
    random_state=42
):
    rng = np.random.default_rng(random_state)

    sampled_parts = []

    for geom, gdf in df.groupby(geometry_col):

        pareto_df = gdf[gdf[pareto_col]]
        nonpareto_df = gdf[~gdf[pareto_col]]

        n_pareto = len(pareto_df)
        n_non = len(nonpareto_df)

        n_sample = min(
            samples_per_class_per_geometry,
            n_pareto,
            n_non
        )

        if n_sample == 0:
            print(f"Skipping {geom}: not enough Pareto/non-Pareto points.")
            continue

        sampled_pareto = pareto_df.sample(
            n=n_sample,
            random_state=random_state
        )

        sampled_nonpareto = nonpareto_df.sample(
            n=n_sample,
            random_state=random_state
        )

        sampled_parts.append(sampled_pareto)
        sampled_parts.append(sampled_nonpareto)

        print(
            f"{geom}: sampled {n_sample} Pareto and {n_sample} non-Pareto"
        )

    balanced_df = pd.concat(
        sampled_parts,
        ignore_index=True
    )

    balanced_df["Pareto_Label"] = balanced_df[pareto_col].astype(int)

    return balanced_df


# ============================================================
# 6. Train DT + RF classifiers
# ============================================================

def train_pareto_classifiers(
    balanced_df,
    classifier_features,
    label_col="Pareto_Label",
    test_size=0.30,
    random_state=42,
    dt_max_depth=3
):
    X_cls = balanced_df[classifier_features].copy()
    y_cls = balanced_df[label_col].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X_cls,
        y_cls,
        test_size=test_size,
        random_state=random_state,
        stratify=y_cls
    )

    dt_model = DecisionTreeClassifier(
        max_depth=dt_max_depth,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=random_state
    )

    dt_model.fit(X_train, y_train)

    rf_cls = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1
    )

    rf_cls.fit(X_train, y_train)

    print("\nDecision Tree classifier:")
    print(classification_report(y_test, dt_model.predict(X_test)))

    print("\nRandom Forest classifier:")
    print(classification_report(y_test, rf_cls.predict(X_test)))

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "dt_model": dt_model,
        "rf_classifier": rf_cls
    }


# ============================================================
# 7. Plot decision tree
# ============================================================

from sklearn.tree import _tree, plot_tree
import numpy as np
import matplotlib.pyplot as plt

def plot_pareto_decision_tree_clean(
    dt_model,
    feature_names,
    save_path="Pareto_Decision_Tree_Clean.png",
    figsize=(22, 9),
    dpi=300,
    font_family="Times New Roman",
    font_size=24,
    line_width=2.4,
    pareto_color="#EE6A50"
):
    plt.rcParams["font.family"] = font_family

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    annotations = plot_tree(
        dt_model,
        feature_names=feature_names,
        class_names=["Dominated", "Pareto"],
        filled=False,
        rounded=True,
        impurity=False,
        proportion=False,
        precision=3,
        fontsize=font_size,
        ax=ax
    )

    tree = dt_model.tree_

    for node_id, ann in enumerate(annotations):
        bbox = ann.get_bbox_patch()

        if bbox is None:
            continue

        is_leaf = (
            tree.children_left[node_id] == _tree.TREE_LEAF and
            tree.children_right[node_id] == _tree.TREE_LEAF
        )

        predicted_class = np.argmax(tree.value[node_id][0])

        if is_leaf and predicted_class == 1:
            bbox.set_facecolor(pareto_color)
            bbox.set_alpha(0.90)
        else:
            bbox.set_facecolor("white")
            bbox.set_alpha(1.0)

        bbox.set_edgecolor("black")
        bbox.set_linewidth(line_width)

        ann.set_fontsize(font_size)
        ann.set_color("black")

    for artist in ax.get_children():
        if hasattr(artist, "set_linewidth"):
            try:
                artist.set_linewidth(line_width)
            except Exception:
                pass

    ax.axis("off")
    plt.tight_layout()

    plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")

    plt.show()

# ============================================================
# 8. Plot permutation importance for RF classifier
# ============================================================

def plot_pareto_classifier_importance(
    rf_classifier,
    X_test,
    y_test,
    feature_label_map=None,
    top_n=12,
    save_path="Pareto_RF_Classifier_Importance.png",
    figsize=(7.2, 5.2),
    dpi=300,
    font_family="Times New Roman",
    font_size=18,
    bar_color="#EE6A50"
):
    plt.rcParams["font.family"] = font_family

    if feature_label_map is None:
        feature_label_map = {}

    perm = permutation_importance(
        rf_classifier,
        X_test,
        y_test,
        n_repeats=20,
        random_state=42,
        n_jobs=-1
    )

    imp_df = pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": perm.importances_mean,
        "Importance_STD": perm.importances_std
    })

    imp_df = imp_df.sort_values(
        "Importance",
        ascending=False
    ).head(top_n)

    imp_df["Feature_Display"] = imp_df["Feature"].map(
        lambda x: feature_label_map.get(x, x)
    )

    imp_df = imp_df.sort_values(
        "Importance",
        ascending=True
    )

    fig, ax = plt.subplots(
        figsize=figsize,
        dpi=dpi
    )

    ax.barh(
        imp_df["Feature_Display"],
        imp_df["Importance"],
        xerr=imp_df["Importance_STD"],
        color=bar_color,
        edgecolor="black",
        linewidth=1.1,
        error_kw=dict(
            linewidth=1.0,
            capsize=3,
            capthick=1.0
        )
    )

    ax.set_xlabel(
        "Permutation importance for Pareto classification",
        fontsize=font_size
    )

    ax.tick_params(
        axis="both",
        labelsize=font_size - 2,
        direction="in",
        length=6,
        width=1.2
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(nbins=5, min_n_ticks=4)
    )

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-1, 2))
    formatter.set_useOffset(False)
    ax.xaxis.set_major_formatter(formatter)

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_linewidth(1.3)
        spine.set_color("black")

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight"
    )

    plt.savefig(
        save_path.replace(".png", ".pdf"),
        bbox_inches="tight"
    )

    plt.show()

    return imp_df


# ============================================================
# 9. Optional SHAP summary for RF Pareto classifier
# ============================================================

def try_plot_shap_for_pareto_classifier(
    rf_classifier,
    X_sample,
    feature_label_map=None,
    save_path="Pareto_RF_Classifier_SHAP.png"
):
    try:
        import shap

        X_plot = X_sample.copy()

        if feature_label_map is not None:
            X_plot = X_plot.rename(columns=feature_label_map)

        explainer = shap.TreeExplainer(rf_classifier)
        shap_values = explainer.shap_values(X_sample)

        # For binary classifier, class 1 = Pareto
        if isinstance(shap_values, list):
            shap_values_to_plot = shap_values[1]
        else:
            shap_values_to_plot = shap_values

        shap.summary_plot(
            shap_values_to_plot,
            X_plot,
            show=False
        )

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")
        plt.show()

    except Exception as e:
        print("SHAP plot skipped because SHAP failed or is not installed.")
        print("Error:", e)

In [ ]:
# ============================================================
# Publication-quality Decision Tree using Graphviz
# Clean rules + coral Pareto leaves
# ============================================================

import numpy as np
from sklearn.tree import _tree
from graphviz import Digraph


def export_clean_pareto_tree_graphviz(
    dt_model,
    feature_names,
    save_name="Pareto_Decision_Tree_Graphviz",
    class_names=("Dominated", "Pareto"),
    pareto_color="#EE6A50",
    font_name="Times New Roman",
    font_size=24,
    node_sep=0.85,
    rank_sep=0.95,
    pen_width=2.4,
    precision=3
):
    tree = dt_model.tree_

    dot = Digraph(
        name="ParetoDecisionTree",
        format="pdf"
    )

    dot.attr(
        rankdir="TB",
        splines="line",
        nodesep=str(node_sep),
        ranksep=str(rank_sep),
        bgcolor="white"
    )

    dot.attr(
        "node",
        shape="box",
        style="rounded,filled",
        fontname=font_name,
        fontsize=str(font_size),
        penwidth=str(pen_width),
        color="black",
        margin="0.18,0.10"
    )

    dot.attr(
        "edge",
        color="black",
        penwidth=str(pen_width),
        fontname=font_name,
        fontsize=str(font_size - 4)
    )

    def node_label(node_id):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        # Leaf node
        if left == _tree.TREE_LEAF and right == _tree.TREE_LEAF:
            counts = tree.value[node_id][0]
            pred_class = int(np.argmax(counts))
            prob = counts[pred_class] / counts.sum()

            return f"{class_names[pred_class]}\\n{100*prob:.0f}%"

        # Split node
        feature_id = tree.feature[node_id]
        threshold = tree.threshold[node_id]
        feature_name = feature_names[feature_id]

        return f"{feature_name} ≤ {threshold:.{precision}g}"

    def add_node(node_id):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        counts = tree.value[node_id][0]
        pred_class = int(np.argmax(counts))

        is_leaf = (
            left == _tree.TREE_LEAF and
            right == _tree.TREE_LEAF
        )

        if is_leaf and pred_class == 1:
            fillcolor = pareto_color
        else:
            fillcolor = "white"

        dot.node(
            str(node_id),
            label=node_label(node_id),
            fillcolor=fillcolor
        )

        if left != _tree.TREE_LEAF:
            add_node(left)
            dot.edge(str(node_id), str(left), label="Yes")

        if right != _tree.TREE_LEAF:
            add_node(right)
            dot.edge(str(node_id), str(right), label="No")

    add_node(0)

    output_path = dot.render(
        filename=save_name,
        cleanup=True
    )

    print(f"Saved: {output_path}")

    return dot

In [ ]:
# ============================================================
# Clean engineering-rule Decision Tree with Graphviz
# ============================================================

import numpy as np
from sklearn.tree import _tree
from graphviz import Digraph


def export_engineering_rule_tree(
    dt_model,
    feature_names,
    save_name="Pareto_Engineering_Rule_Tree",
    class_names=("Dominated", "Pareto"),
    pareto_color="#EE6A50",
    font_name="Times New Roman",
    font_size=24,
    node_sep=0.95,
    rank_sep=1.05,
    pen_width=2.5,
    precision=3
):
    tree = dt_model.tree_

    dot = Digraph(
        name="ParetoEngineeringRuleTree",
        format="pdf"
    )

    dot.attr(
        rankdir="TB",
        splines="ortho",
        nodesep=str(node_sep),
        ranksep=str(rank_sep),
        bgcolor="white"
    )

    dot.attr(
        "node",
        shape="box",
        style="rounded,filled",
        fontname=font_name,
        fontsize=str(font_size),
        penwidth=str(pen_width),
        color="black",
        margin="0.18,0.12"
    )

    dot.attr(
        "edge",
        color="black",
        penwidth=str(pen_width),
        fontname=font_name,
        fontsize=str(font_size - 5),
        arrowsize="0.8"
    )

    def make_node_label(node_id):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        counts = tree.value[node_id][0]
        pred_class = int(np.argmax(counts))
        prob = counts[pred_class] / counts.sum()

        # Leaf node
        if left == _tree.TREE_LEAF and right == _tree.TREE_LEAF:
            return f"{class_names[pred_class]}\\n{100 * prob:.0f}%"

        # Split node
        feature_id = tree.feature[node_id]
        threshold = tree.threshold[node_id]
        feature_name = feature_names[feature_id]

        return f"{feature_name} \\leq {threshold:.{precision}g}"

    def add_node(node_id):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        counts = tree.value[node_id][0]
        pred_class = int(np.argmax(counts))

        is_leaf = (
            left == _tree.TREE_LEAF and
            right == _tree.TREE_LEAF
        )

        if is_leaf and pred_class == 1:
            fillcolor = pareto_color
        else:
            fillcolor = "white"

        dot.node(
            str(node_id),
            label=make_node_label(node_id),
            fillcolor=fillcolor
        )

        if left != _tree.TREE_LEAF:
            add_node(left)
            dot.edge(str(node_id), str(left), label="Yes")

        if right != _tree.TREE_LEAF:
            add_node(right)
            dot.edge(str(node_id), str(right), label="No")

    add_node(0)

    output_path = dot.render(
        filename=save_name,
        cleanup=True
    )

    print(f"Saved: {output_path}")

    return dot

In [ ]:
# ============================================================
# Publication-quality Decision Tree using Graphviz
# Clean rules + coral Pareto leaves + HTML subscripts
# ============================================================

import numpy as np
from sklearn.tree import _tree
from graphviz import Digraph


def export_clean_pareto_tree_graphviz(
    dt_model,
    feature_names,
    save_name="Pareto_Decision_Tree_Graphviz",
    class_names=("Dominated", "Pareto"),
    pareto_color="#EE6A50",
    font_name="Times New Roman",
    font_size=24,
    node_sep=0.85,
    rank_sep=0.95,
    pen_width=2.4,
    precision=3
):
    tree = dt_model.tree_

    dot = Digraph(
        name="ParetoDecisionTree",
        format="pdf"
    )

    dot.attr(
        rankdir="TB",
        splines="line",
        nodesep=str(node_sep),
        ranksep=str(rank_sep),
        bgcolor="white"
    )

    dot.attr(
        "node",
        shape="box",
        style="rounded,filled",
        fontname=font_name,
        fontsize=str(font_size),
        penwidth=str(pen_width),
        color="black",
        margin="0.18,0.10"
    )

    dot.attr(
        "edge",
        color="black",
        penwidth=str(pen_width),
        fontname=font_name,
        fontsize=str(font_size - 4)
    )

    def node_label(node_id):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        # Leaf node
        if left == _tree.TREE_LEAF and right == _tree.TREE_LEAF:
            counts = tree.value[node_id][0]
            pred_class = int(np.argmax(counts))
            prob = counts[pred_class] / counts.sum()

            return (
                f"<<B>{class_names[pred_class]}</B><BR/>"
                f"{100 * prob:.0f}%>"
            )

        # Split node
        feature_id = tree.feature[node_id]
        threshold = tree.threshold[node_id]
        feature_name = feature_names[feature_id]

        return (
            f"<<B>{feature_name}</B><BR/>"
            f"&le; {threshold:.{precision}g}>"
        )

    def add_node(node_id):
        left = tree.children_left[node_id]
        right = tree.children_right[node_id]

        counts = tree.value[node_id][0]
        pred_class = int(np.argmax(counts))

        is_leaf = (
            left == _tree.TREE_LEAF and
            right == _tree.TREE_LEAF
        )

        fillcolor = pareto_color if is_leaf and pred_class == 1 else "white"

        dot.node(
            str(node_id),
            label=node_label(node_id),
            fillcolor=fillcolor
        )

        if left != _tree.TREE_LEAF:
            add_node(left)
            dot.edge(str(node_id), str(left), label="Yes")

        if right != _tree.TREE_LEAF:
            add_node(right)
            dot.edge(str(node_id), str(right), label="No")

    add_node(0)

    output_path = dot.render(
        filename=save_name,
        cleanup=True
    )

    print(f"Saved: {output_path}")

    return dot

## 8. Example analyses used to reproduce manuscript figures

The example cells below can be adjusted to reproduce a specific manuscript figure. The `fixed_geometry` dictionary controls the building class, and `custom_feature_ranges` controls design-variable sampling ranges.

In [ ]:
# ============================================================
# Example: 19-story, 30-ft bay width Sobol analysis
# ============================================================

fixed_geometry = {
    "number of stories (NS)": 19,
    "bay width (BW), ft": 30
}

sobol_results_df, sobol_prediction_df, sobol_problem = run_sobol_rf_fixed_geometry(
    X_original=X,
    rf_models=rf_models,
    target_cols=target_cols,
    fixed_geometry=fixed_geometry,
    feature_cols=None,
    base_sample_size=1024,
    range_quantiles=(0.05, 0.95),
    calc_second_order=False
)

display(sobol_results_df.head(30))

In [ ]:
geometry_features = [
    "total building height (TBH), ft",
    "Total building height/first story height",
]
for target in target_cols:

    plot_sobol_indices(
        sobol_results_df=sobol_results_df,
        target=target,
        top_n=10,
        feature_name_map=feature_name_map,
        exclude_features=geometry_features,
        index_type="ST",
        title=target,
        save_path=f"Sobol_ST_19S_{target}.png"
    )

In [ ]:
fixed_geometry = {
    "number of stories (NS)": 1,
    "bay width (BW), ft": 30
}

tradeoff_feature_ranges = {
    "Avg_ColumnBeamRatio (AVG_SCWB)": (1.0, 2.0)
}

tradeoff_df, X_syn, X_filtered = run_fixed_geometry_tradeoff_analysis(
    X_original=X,
    rf_models=rf_models,
    fixed_geometry=fixed_geometry,
    color_col="Avg_ColumnBeamRatio (AVG_SCWB)",
    cbar_label="Average SCWB",
    custom_feature_ranges=tradeoff_feature_ranges,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    save_prefix="1s_30BW_SCWB_Tradeoff"
)

In [ ]:
fixed_geometry = {
    "number of stories (NS)": 19,
    "bay width (BW), ft": 30
}


selected_ale_features = [
    "Avg_ColumnBeamRatio (AVG_SCWB)",
    "Avg_DgnStoryDrift",
    "TimePeriod",
    "Building FRAME weight (lbf)",
    "Total building weight (lbf)"
]

#custom_feature_ranges = {
#    "Avg_ColumnBeamRatio (AVG_SCWB)": (1.0, 2.0)
#}

target_label_map = {
    "Functional_DBE": "DBE recovery",
    "Functional_MCE": "MCE recovery",
    "EAL": "EAL"
}


X_ale, X_filtered_ale = run_fixed_geometry_combined_ale(
    X_original=X,
    rf_models=rf_models,
    fixed_geometry=fixed_geometry,
    selected_features=selected_ale_features,
    target_cols=["Functional_DBE", "Functional_MCE", "EAL"],

    custom_feature_ranges=custom_feature_ranges,   # <-- THIS LINE

    feature_name_map=feature_name_map,
    target_label_map=target_label_map,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05,0.95),
    grid_size=35,
    smooth=True,
    smooth_sigma=1.0,
    save_prefix="Combined_ALE_19S_30BW")

In [ ]:
run_tradeoff_maps_for_top_sobol_features(
    sobol_results_df=sobol_results_df,
    sobol_prediction_df=sobol_prediction_df,
    target_for_ranking="EAL",
    top_n=3,
    index_type="ST",
    exclude_features=geometry_features,
    feature_name_map=feature_name_map,
    save_prefix="Tradeoff_19S_30BW"
)

In [ ]:
fixed_geometry = {
    "number of stories (NS)": 1,
    "bay width (BW), ft": 30
}

#custom_feature_ranges = {
 #   "Avg_ColumnBeamRatio (AVG_SCWB)": (1.0, 3.0),
  #  "Avg_DgnStoryDrift": (0.001, 0.01)
#}

pareto_df, X_syn, X_filtered = run_fixed_geometry_pareto_3d(
    X_original=X,
    rf_models=rf_models,
    fixed_geometry=fixed_geometry,
    custom_feature_ranges=custom_feature_ranges,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    save_prefix="Pareto_1S_30BW",
    show_legend=False,
    elev=24,
    azim=-55
)

In [ ]:
pareto_feature_cols = [
    "Avg_ColumnBeamRatio (AVG_SCWB)",
    "Avg_DgnStoryDrift",
    "Building FRAME weight (lbf)",
    "TimePeriod"
]

pareto_feature_label_map = {
    "Avg_ColumnBeamRatio (AVG_SCWB)": "SCWB",
    "Avg_DgnStoryDrift": "IDR",
    "Building FRAME weight (lbf)": r"$W_L$",
    "TimePeriod": r"$T_1$"
}

plot_pareto_vs_nonpareto_boxplot(
    df=pareto_df,
    feature_cols=pareto_feature_cols,
    feature_label_map=pareto_feature_label_map,
    standardize=True,
    save_path="Pareto_vs_NonPareto_Boxplot.png"
)

enrichment_summary = plot_pareto_feature_enrichment(
    df=pareto_df,
    feature_cols=pareto_feature_cols,
    feature_label_map=pareto_feature_label_map,
    metric="Percent_Change",
    save_path="Pareto_Feature_Enrichment_PercentChange.png"
)

display(enrichment_summary)

In [ ]:
# ============================================================
# Example use
# ============================================================

building_cases = {
    "1S_30BW": {
        "number of stories (NS)": 1,
        "bay width (BW), ft": 30
    },
    "1S_20BW": {
        "number of stories (NS)": 1,
        "bay width (BW), ft": 20
    },
    "5S_30BW": {
        "number of stories (NS)": 5,
        "bay width (BW), ft": 30
    },
    "5S_20BW": {
        "number of stories (NS)": 5,
        "bay width (BW), ft": 20
    },
    "9S_30BW": {
        "number of stories (NS)": 9,
        "bay width (BW), ft": 30
    },
    "9S_20BW": {
        "number of stories (NS)": 9,
        "bay width (BW), ft": 20
    },
    "14S_30BW": {
        "number of stories (NS)": 14,
        "bay width (BW), ft": 30
    },
    "14S_20BW": {
        "number of stories (NS)": 14,
        "bay width (BW), ft": 20
    },
    "19S_30BW": {
        "number of stories (NS)": 19,
        "bay width (BW), ft": 30
    },
     "19S_20BW": {
        "number of stories (NS)": 19,
        "bay width (BW), ft": 20
    }
}

custom_feature_ranges = {
    "Avg_ColumnBeamRatio (AVG_SCWB)": (1.0, 3.0),
    "Avg_DgnStoryDrift": (0.001, 0.01)
}

all_pareto_df = build_multi_geometry_pareto_dataset(
    X_original=X,
    rf_models=rf_models,
    building_cases=building_cases,
    custom_feature_ranges=custom_feature_ranges,
    n_random_samples=50000,
    random_state=42,
    range_quantiles=(0.05, 0.95),
    objective_cols=("Functional_DBE", "Functional_MCE", "EAL")
)

balanced_pareto_df = create_balanced_pareto_classifier_dataset(
    df=all_pareto_df,
    samples_per_class_per_geometry=100,
    pareto_col="Pareto_Efficient",
    geometry_col="Geometry_Case",
    random_state=42
)

classifier_features = [
    "Avg_ColumnBeamRatio (AVG_SCWB)",
    "Avg_DgnStoryDrift",
    "Building FRAME weight (lbf)",
    "Total building weight (lbf)",
    "TimePeriod"
]

classifier_features = [
    col for col in classifier_features
    if col in balanced_pareto_df.columns
]

feature_label_map = {
    "Avg_ColumnBeamRatio (AVG_SCWB)": "SCWB",
    "Avg_DgnStoryDrift": "IDR",
    "Building FRAME weight (lbf)": r"$W_L$",
    "Total building weight (lbf)": r"$W_T$",
    "TimePeriod": r"$T_1$"
}

models = train_pareto_classifiers(
    balanced_df=balanced_pareto_df,
    classifier_features=classifier_features,
    test_size=0.30,
    random_state=42,
    dt_max_depth=3
)

plot_pareto_decision_tree_clean(
    dt_model=models["dt_model"],
    feature_names=[feature_label_map.get(f, f) for f in classifier_features],
    save_path="Pareto_Decision_Tree_Clean.png"
)

importance_df = plot_pareto_classifier_importance(
    rf_classifier=models["rf_classifier"],
    X_test=models["X_test"],
    y_test=models["y_test"],
    feature_label_map=feature_label_map,
    top_n=10,
    save_path="Pareto_RF_Classifier_Importance.png"
)

display(importance_df)

try_plot_shap_for_pareto_classifier(
    rf_classifier=models["rf_classifier"],
    X_sample=models["X_test"].sample(
        min(500, len(models["X_test"])),
        random_state=42
    ),
    feature_label_map=feature_label_map,
    save_path="Pareto_RF_Classifier_SHAP.png"
)

In [ ]:
# ------------------------------------------------------------
# Feature names for Graphviz
# (HTML labels allow proper subscripts)
# ------------------------------------------------------------

feature_label_map = {
    "Avg_ColumnBeamRatio (AVG_SCWB)": "SCWB<SUB>avg</SUB>",
    "Avg_DgnStoryDrift": "IDR<SUB>des</SUB>",
    "Building FRAME weight (lbf)": "W<SUB>L</SUB>",
    "Total building weight (lbf)": "W<SUB>T</SUB>",
    "TimePeriod": "T<SUB>1</SUB>"
}

feature_names_clean = [
    feature_label_map.get(f, f)
    for f in classifier_features
]

# ------------------------------------------------------------
# Export publication-quality decision tree
# ------------------------------------------------------------

dot = export_clean_pareto_tree_graphviz(
    dt_model=models["dt_model"],
    feature_names=feature_names_clean,
    save_name="Pareto_Decision_Tree_Graphviz",
    class_names=("Dominated", "Pareto"),
    pareto_color="#EE6A50",
    font_name="Times New Roman",
    font_size=26,      # increase if desired
    node_sep=1.2,      # horizontal spacing
    rank_sep=1.2,      # vertical spacing
    pen_width=2.8,     # thicker borders
    precision=3
)

# Display inside Jupyter
dot

## Notes for GitHub/Zenodo release

Before archiving this repository, add:
1. `requirements.txt` with the exact Python package versions.
2. A README table mapping each manuscript figure/table to the notebook cell or output file that reproduces it.
3. Processed training and test data, or a clear statement identifying any files that cannot be shared and the reason.
4. A release through Zenodo to obtain a DOI.